In [ ]:
import os

os.chdir("../")
from gravity import Gravity, Production, Attraction, Doubly

os.chdir("/Users/toshan/dev/pysal/pysal/contrib/spint")
import entropy as grav
import numpy as np
import scipy.stats as stats
import pandas as pd
import seaborn as sns
import geopandas as gp

os.chdir("/Users/toshan/Dropbox/Data/NYC_BIKES")
import matplotlib.pylab as plt

%pylab inline
from descartes import PolygonPatch
import matplotlib as mpl
from mpl_toolkits.basemap import Basemap
import pyproj as pj
from shapely.geometry import Polygon, Point

In [ ]:
ct = pd.read_csv("CT_BIKE_DATA.csv")
ct = ct[ct["o_tract"] != ct["d_tract"]]

ct.ix[ct.o_sq_foot == 0, "o_sq_foot"] = 1
ct.ix[ct.d_sq_foot == 0, "d_sq_foot"] = 1
ct.ix[ct.o_cap == 0, "o_cap"] = 1
ct.ix[ct.d_cap == 0, "d_cap"] = 1
ct.ix[ct.o_housing == 0, "o_housing"] = 1
ct.ix[ct.d_housing == 0, "d_housing"] = 1

flows = ct["count"].values.reshape((-1, 1))
o_vars = np.hstack(
    [
        ct["o_sq_foot"].values.reshape((-1, 1)),
        ct["o_housing"].values.reshape((-1, 1)),
        ct["o_cap"].values.reshape((-1, 1)),
    ]
)
d_vars = np.hstack(
    [
        ct["d_sq_foot"].values.reshape((-1, 1)),
        ct["d_housing"].values.reshape((-1, 1)),
        ct["d_cap"].values.reshape((-1, 1)),
    ]
)
cost = ct["tripduration"].values.reshape((-1, 1))
o = ct["o_tract"].astype(str).values.reshape((-1, 1))
d = ct["d_tract"].astype(str).values.reshape((-1, 1))
print(len(ct), " OD pairs between census tracts after filtering out intrazonal flows")
ct.head()

In [ ]:
os.chdir("/Users/toshan/dev/pysal/pysal/contrib/spint")
from gravity import Gravity, Production, Attraction, Doubly

model = Gravity(flows, o_vars, d_vars, cost, "exp")
print(model.params)
print(model.deviance)

In [ ]:
model = Production(flows, o, d_vars, cost, "exp")
print(model.params[-4:])
print(model.deviance)

In [ ]:
model = Doubly(flows, o, d, cost, "exp")
print(model.params[-1])
print(model.deviance)

In [ ]:
model = Production(flows, o, d_vars, cost, "exp")
local = model.local()

In [ ]:
local["param4"][:10]

In [ ]:
model.params[-1:] - local["param4"]

In [ ]:
crs = {"datum": "WGS84", "proj": "longlat"}
tracts = gp.read_file("/Users/toshan/Dropbox/Data/NYC_BIKES/nyct2010_15a/nyct2010.shp")
tracts = tracts.to_crs(crs=crs)
man_tracts = tracts[tracts["BoroCode"] == "1"].copy()
man_tracts["CT2010S"] = man_tracts["CT2010"].astype(int).astype(str)
mt = set(man_tracts.CT2010S.unique())
lt = set(np.unique(o))
nt = list(mt.difference(lt))
no_tracts = pd.DataFrame({"no_tract": nt})
no_tracts = man_tracts[man_tracts.CT2010S.isin(nt)].copy()
local_vals = pd.DataFrame({"betas": local["param4"], "tract": np.unique(o)})
local_vals = pd.merge(
    local_vals, man_tracts[["CT2010S", "geometry"]], left_on="tract", right_on="CT2010S"
)
local_vals = gp.GeoDataFrame(local_vals)

In [ ]:
plt.figure(figsize=(12, 12))
local_vals["inv_betas"] = local_vals["betas"] * -1
no_tracts["test"] = 0
no_tracts.plot("test", colormap="copper")
local_vals.plot("inv_betas", colormap="Blues")
plt.xlim(-74.02, -73.95)
plt.ylim(40.7, 40.78)

In [ ]:
plt.figure(figsize=(12, 12))
local_vals["cap"] = local["param3"]
no_tracts["test"] = 0
no_tracts.plot("test", colormap="copper")
local_vals.plot("cap", colormap="Reds")
plt.legend()
plt.xlim(-74.02, -73.95)
plt.ylim(40.7, 40.78)

In [ ]:
plt.figure(figsize=(12, 12))
local_vals["house"] = local["param2"]
no_tracts["test"] = 0
no_tracts.plot("test", colormap="copper")
local_vals.plot("house", colormap="Reds")
plt.legend()
plt.xlim(-74.02, -73.95)
plt.ylim(40.7, 40.78)

In [ ]:
plt.figure(figsize=(12, 12))
local_vals["foot"] = local["param1"]
no_tracts["test"] = 0
no_tracts.plot("test", colormap="copper")
local_vals.plot("foot", colormap="Reds")
plt.legend()
plt.xlim(-74.02, -73.95)
plt.ylim(40.7, 40.78)